# Jakarta Q8 strain-rate inversion

A Python implementation to estimate horizontal strain-rate tensors and derived strain-rate quantities from InSAR-based regional uniform-motion (RUM) horizontal velocities, applied over Jakarta. The method uses area-average RUM velocity observations, Q8 quadratic quadrilateral finite elements, regularized inversion, and first-order covariance propagation.

This notebook provides the implementation of the horizontal strain-rate inversion method introduced in the accompanying manuscript:
<br>
&emsp; Lapadat and Hanssen, *Horizontal Strain Rate Estimation from InSAR Observations and the Strapdown Method*. <br>&emsp; Manuscript submitted to *Journal of Geophysical Research: Solid Earth*.
<br>
It loads the RUM velocities, either performs the inversion from scratch or loads precomputed inversion results, 
derives the strain-rate quantities and their within-element covariance blocks, 
and exports the numerical results;

**Note:** Running the complete strain-rate inversion is computationally expensive. The notebook therefore provides the option to load the precomputed inversion results from `results/jakarta_strain_inversion_archive.pkl`. This allows users to bypass the inversion and proceed directly with deriving the strain-rate quantities, exporting the result tables, and reproducing the manuscript figures.


In [1]:
# ============================================================
# --- Imports, folders, workflow settings, and data preparation ---
# ============================================================

from html import unescape
from pathlib import Path
from urllib.parse import urljoin
import csv
import hashlib
import json
import pickle
import re
import requests
import numpy as np
import pandas as pd

## IMPORTANT OPTION: ENABLING/DISABLING INVERSION ##
# False: load the distributed archive. True: execute the full inversion.
RUN_FULL_INVERSION = False

# Public strain-rate dataset.
STRAIN_DATASET_URL = ("https://data.4tu.nl/datasets/cff30595-1d98-4e9d-8eb7-0a3000a434fe")

# Public RUM velocity file and its repository-provided checksum.
INPUT_CSV_URL = (
    "https://data.4tu.nl/file/"
    "39e7a39c-d305-425f-b238-1f3a4271dda9/"
    "178abd13-6f3a-49c3-bbce-16588e28b98b"
)
INPUT_CSV_MD5 = "1fc1fbfd8815c4ab457c8cdfc67fbd18"

# Find the repository root whether the notebook is opened from the root
# directory or from a notebook subdirectory.
ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
for directory in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Input files used by the inversion and figure notebooks.
INPUT_CSV = DATA_DIR / "jakarta_enu_estimates.csv"
CONTOUR_NPZ_PATH = DATA_DIR / "jakarta_transv_horiz_vel_contours.npz"

# Precomputed inversion products and public output tables.
ARCHIVE_PATH = RESULTS_DIR / "jakarta_strain_inversion_archive.pkl"
TENSOR_CSV_PATH = RESULTS_DIR / "jakarta_strain_tensor.csv"
QUANTITY_CSV_PATH = RESULTS_DIR / "jakarta_strain_quantities.csv"
METADATA_JSON_PATH = RESULTS_DIR / "jakarta_strain_metadata.json"

# Network requests use separate connection and read timeouts. The longer read
# timeout accommodates the large precomputed inversion archive.
NETWORK_TIMEOUT = (15, 300)
DOWNLOAD_CHUNK_SIZE = 1024 * 1024


# ============================================================
# --- Download and validation helpers ---
# ============================================================

def md5(path):
    """Return the MD5 checksum of a local file without loading it into memory."""
    checksum = hashlib.md5()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(DOWNLOAD_CHUNK_SIZE), b""):
            checksum.update(block)
    return checksum.hexdigest()


def relative_path(path):
    """Return a repository-relative path for concise status messages."""
    return Path(path).resolve().relative_to(ROOT).as_posix()


def validate_downloaded_file(path, expected_md5=None):
    """Reject empty, HTML, malformed, or checksum-mismatched downloads."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError("the downloaded file is empty")

    with path.open("rb") as stream:
        prefix = stream.read(4096)
    prefix_lower = prefix.lstrip().lower()
    if prefix_lower.startswith((b"<!doctype html", b"<html")):
        raise ValueError("4TU returned an HTML page instead of the requested file")

    if expected_md5 is not None:
        received_md5 = md5(path)
        if received_md5.lower() != expected_md5.lower():
            raise ValueError(
                f"checksum mismatch: expected {expected_md5}, received {received_md5}"
            )

    suffix = path.suffix.lower()
    if suffix == ".csv":
        with path.open("r", encoding="utf-8-sig", newline="") as stream:
            header = next(csv.reader(stream), [])
        if len(header) < 2:
            raise ValueError("the downloaded CSV does not contain a valid header")
    elif suffix == ".json":
        with path.open("r", encoding="utf-8") as stream:
            json.load(stream)
    elif suffix == ".npz" and not prefix.startswith(b"PK"):
        raise ValueError("the downloaded NPZ does not have a ZIP/NPZ signature")
    # The PKL archive is intentionally not unpickled during download validation.


def download_file(session, url, destination, expected_md5=None, headers=None):
    """Stream one file to a temporary path and publish it only after validation."""
    destination = Path(destination)
    temporary_path = destination.with_name(destination.name + ".download")
    temporary_path.unlink(missing_ok=True)

    try:
        with session.get(
            url,
            headers=headers,
            stream=True,
            timeout=NETWORK_TIMEOUT,
        ) as response:
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "").lower()
            if "text/html" in content_type:
                raise ValueError(
                    "4TU returned HTML instead of a file; the sharing link may "
                    "be unavailable or expired"
                )

            with temporary_path.open("xb") as stream:
                for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                    if chunk:
                        stream.write(chunk)

        validate_downloaded_file(temporary_path, expected_md5=expected_md5)
        if destination.exists():
            raise FileExistsError(
                f"refusing to overwrite an existing file: {relative_path(destination)}"
            )
        temporary_path.replace(destination)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise


# ============================================================
# --- Resolve files from the 4TU strain-rate dataset page ---
# ============================================================

def resolve_strain_dataset_files(session, required_names):
    """Read the 4TU dataset page and return each required file URL and MD5."""
    if STRAIN_DATASET_URL == "REPLACE_WITH_PUBLIC_4TU_DATASET_URL":
        raise ValueError(
            "STRAIN_DATASET_URL must be replaced with the published 4TU "
            "strain-rate dataset URL"
        )
    response = session.get(STRAIN_DATASET_URL, timeout=NETWORK_TIMEOUT)
    response.raise_for_status()

    page_html = response.text
    content_type = response.headers.get("Content-Type", "").lower()
    page_start = page_html.lstrip().lower()
    is_html = (
        "html" in content_type
        or page_start.startswith("<!doctype html")
        or page_start.startswith("<html")
    )
    if not is_html:
        raise ValueError("the strain-rate dataset URL did not return an HTML dataset page")

    # A 4TU dataset page lists every file in an HTML list item. Each item
    # contains the repository-provided MD5 and its individual /file/... link.
    file_records = {}
    list_items = re.findall(
        r"<li\b[^>]*>(.*?)</li>",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )
    for item in list_items:
        link_match = re.search(
            r"""<a\b[^>]*href=["']([^"']*/file/[^"']+)["'][^>]*>(.*?)</a>""",
            item,
            flags=re.IGNORECASE | re.DOTALL,
        )
        if link_match is None:
            continue

        filename = re.sub(r"<[^>]+>", "", link_match.group(2))
        filename = unescape(filename).strip()
        checksum_match = re.search(
            r"MD5:\s*<code[^>]*>\s*([0-9a-f]{32})\s*</code>",
            item,
            flags=re.IGNORECASE,
        )
        file_records[filename] = {
            "url": urljoin(response.url, unescape(link_match.group(1))),
            "md5": checksum_match.group(1).lower() if checksum_match else None,
        }

    missing_names = sorted(set(required_names) - set(file_records))
    if missing_names:
        raise ValueError(
            "the strain-rate dataset page did not expose the required file links: "
            + ", ".join(missing_names)
        )

    missing_checksums = [
        name for name in required_names if not file_records[name]["md5"]
    ]
    if missing_checksums:
        raise ValueError(
            "the strain-rate dataset page did not expose MD5 checksums for: "
            + ", ".join(missing_checksums)
        )
    return file_records


# ============================================================
# --- Check and prepare all required data files ---
# ============================================================

def prepare_required_data():
    """Check all six required files and download only those that are missing."""
    print("Checking required data files...\n")
    failures = {}

    session = requests.Session()
    session.headers.update({
        "User-Agent": "JakartaStrainResearch/1.0 "
        "(+https://doi.org/10.4121/39e7a39c-d305-425f-b238-1f3a4271dda9.v3)"
    })

    # Preserve the existing public RUM checksum verification. An existing file
    # that fails validation is reported but never overwritten automatically.
    if INPUT_CSV.exists():
        local_md5 = md5(INPUT_CSV)
        if local_md5 == INPUT_CSV_MD5:
            print(f"EXISTS:     {relative_path(INPUT_CSV)}")
        else:
            reason = (
                f"existing file failed checksum validation: expected "
                f"{INPUT_CSV_MD5}, received {local_md5}; it was not overwritten"
            )
            failures[INPUT_CSV] = reason
            print(f"INVALID:    {relative_path(INPUT_CSV)}")
    else:
        try:
            download_file(
                session,
                INPUT_CSV_URL,
                INPUT_CSV,
                expected_md5=INPUT_CSV_MD5,
            )
            print(f"DOWNLOADED: {relative_path(INPUT_CSV)}")
        except Exception as error:
            failures[INPUT_CSV] = str(error)
            print(f"FAILED:     {relative_path(INPUT_CSV)}")

    strain_targets = {
        "jakarta_transv_horiz_vel_contours.npz": CONTOUR_NPZ_PATH,
        "jakarta_strain_tensor.csv": TENSOR_CSV_PATH,
        "jakarta_strain_quantities.csv": QUANTITY_CSV_PATH,
        "jakarta_strain_metadata.json": METADATA_JSON_PATH,
        "jakarta_strain_inversion_archive.pkl": ARCHIVE_PATH,
    }

    missing_strain_names = [
        filename
        for filename, destination in strain_targets.items()
        if not destination.exists()
    ]
    strain_files = None
    strain_resolution_error = None
    if missing_strain_names:
        try:
            strain_files = resolve_strain_dataset_files(
                session,
                missing_strain_names,
            )
        except Exception as error:
            strain_resolution_error = str(error)

    strain_headers = {"Referer": STRAIN_DATASET_URL}
    for filename, destination in strain_targets.items():
        if destination.exists():
            print(f"EXISTS:     {relative_path(destination)}")
            continue

        if strain_resolution_error is not None:
            failures[destination] = strain_resolution_error
            print(f"FAILED:     {relative_path(destination)}")
            continue

        file_record = strain_files[filename]
        try:
            download_file(
                session,
                file_record["url"],
                destination,
                expected_md5=file_record["md5"],
                headers=strain_headers,
            )
            print(f"DOWNLOADED: {relative_path(destination)}")
        except Exception as error:
            failures[destination] = str(error)
            print(f"FAILED:     {relative_path(destination)}")

    required_paths = [INPUT_CSV, *strain_targets.values()]
    missing_paths = [path for path in required_paths if not path.exists()]
    if failures or missing_paths:
        details = []
        for path in required_paths:
            if path in failures:
                details.append(f"- {relative_path(path)}: {failures[path]}")
            elif not path.exists():
                details.append(f"- {relative_path(path)}: file is still missing")
        raise RuntimeError(
            "Data preparation did not complete successfully:\n" + "\n".join(details)
        )

    print("\nData preparation completed successfully.")


prepare_required_data()


Checking required data files...

DOWNLOADED: data/jakarta_enu_estimates.csv
DOWNLOADED: data/jakarta_transv_horiz_vel_contours.npz
DOWNLOADED: results/jakarta_strain_tensor.csv
DOWNLOADED: results/jakarta_strain_quantities.csv
DOWNLOADED: results/jakarta_strain_metadata.json
DOWNLOADED: results/jakarta_strain_inversion_archive.pkl

Data preparation completed successfully.


In [2]:
# ============================================================
# --- Load and validate the public RUM velocity data ---
# ============================================================

# Check the required variables and the expected number of RUM observations.
public_df = pd.read_csv(INPUT_CSV)
required_columns = {
    "x_rum", "y_rum", "east", "north", "up",
    "var_east", "var_north", "var_up",
    "covar_en", "covar_eu", "covar_nu",
}
missing = sorted(required_columns - set(public_df.columns))
if missing:
    raise ValueError(f"Missing public input columns: {missing}")
if len(public_df) != 4779:
    raise ValueError(f"Expected 4779 RUMs, found {len(public_df)}")
print(f"Validated {len(public_df)} public RUM observations.")

# Prepare the in-memory arrays required by the inversion directly from the
# published RUM table. Standard deviations are recovered from the variances.
E_rum_input = public_df["x_rum"].to_numpy(dtype=float)
N_rum_input = public_df["y_rum"].to_numpy(dtype=float)
y_E_input = public_df["east"].to_numpy(dtype=float)
y_N_input = public_df["north"].to_numpy(dtype=float)

var_y_E = public_df["var_east"].to_numpy(dtype=float)
var_y_N = public_df["var_north"].to_numpy(dtype=float)
if np.any(var_y_E <= 0) or np.any(var_y_N <= 0):
    raise ValueError("All published East and North variances must be positive.")

sigma_y_E_input = np.sqrt(var_y_E)
sigma_y_N_input = np.sqrt(var_y_N)
cov_y_EN_input = public_df["covar_en"].to_numpy(dtype=float)


Validated 4779 public RUM observations.


In [3]:
# ============================================================
# --- Optional: load the precomputed inversion archive ---
# ============================================================

if not RUN_FULL_INVERSION:
    with ARCHIVE_PATH.open("rb") as stream:
        archive = pickle.load(stream)
    archive_schema = archive.get("metadata", {}).get("archive_schema")
    if archive_schema != "jakarta_strain_inversion_v2":
        raise ValueError(
            "Expected archive schema jakarta_strain_inversion_v2, "
            f"found {archive_schema!r}."
        )
    archived_xy = np.column_stack([
        archive["mapping"]["E_rum"],
        archive["mapping"]["N_rum"],
    ])
    # The archive and public CSV must describe the same RUMs in the same order.
    public_xy = public_df[["x_rum", "y_rum"]].to_numpy(dtype=float)
    if not np.array_equal(archived_xy, public_xy):
        raise ValueError("Archive RUM ordering/coordinates do not match the public input.")
    print(f"Loaded precomputed inversion archive: {ARCHIVE_PATH.name}")


Loaded precomputed inversion archive: jakarta_strain_inversion_archive.pkl


## Full horizontal strain-rate inversion

The following cell performs the complete inversion only when `RUN_FULL_INVERSION = True`. This calculation is computationally expensive. <br> When `RUN_FULL_INVERSION = False`, the cell is skipped and the distributed inversion archive in `results/jakarta_strain_inversion_archive.pkl` is used instead.


### Inversion settings

The default settings reproduce the Jakarta manuscript result. Changing them produces a different inversion.

#### RUM width and finite-element size

- `RUM_WIDTH_M = 450.0` defines the RUM width and Q8 finite-element spacing in metres.
- For the supplied Jakarta dataset, this must remain `450.0`.
- It should only be changed when using observations representing a different RUM area and grid spacing.

#### Gauss quadrature

- `GAUSS_ORDER = 3` sets the number of Gauss–Legendre integration points along each coordinate direction.
- `3` gives the $3\times3$ quadrature used for the manuscript result.
- Larger integers are allowed but increase computation without being expected to materially change the result for the regular rectangular elements.
- Values below `3` are not recommended for this Q8 formulation.

#### Strain-graph neighborhood

- `STRAIN_GRAPH_RADIUS_M = 3.0 * RUM_WIDTH_M` sets the maximum distance between finite-element centers that can be connected in the strain-rate regularization graph. This is equivalent to the neighboring threshold $d_\mathrm{thresh}$.
- The manuscript value is $1350~\mathrm{m}$, corresponding to three RUM widths.
- A smaller radius makes the regularization more local.
- A larger radius connects more elements, increases the spatial reach of the regularization, and may smooth localized deformation more strongly.
- The radius should be at least one RUM width.

#### Strain-graph weighting

`STRAIN_GRAPH_WEIGHTING` controls how the connections between neighboring elements are weighted:

- `"tensor_sim"` is the manuscript setting. It combines spatial distance with similarity of proxy strain-rate values derived from local fits to the observed RUM velocities.
- `"distance"` uses spatial distance only. Elements at the same separation receive the same initial weight, regardless of their proxy deformation.

#### Distance-weighting kernel

`STRAIN_GRAPH_DISTANCE_KERNEL` controls how graph weights vary with distance:

- `"quadratic"` is the manuscript setting and produces a gradual decrease with distance;
- `"exponential"` produces a stronger decrease with distance;
- `"binary"` assigns the same distance weight to all connected elements.

When `STRAIN_GRAPH_WEIGHTING = "tensor_sim"`, the selected distance weight is subsequently modified using proxy-strain similarity.

#### Proxy-strain comparison

`STRAIN_GRAPH_VOLUMETRIC_ONLY` is used only when `STRAIN_GRAPH_WEIGHTING = "tensor_sim"`:

- `True` is the manuscript setting. Similarity is evaluated using the trace of the proxy strain-rate tensor and therefore emphasizes differences between contraction and dilatation.
- `False` compares the proxy principal strain-rate values. Their orientations are not compared.

#### Strain-similarity scale

`STRAIN_SIMILARITY_SCALE` is used only when `STRAIN_GRAPH_WEIGHTING = "tensor_sim"`:

- `None` is the manuscript setting. The scale is estimated automatically from the median absolute deviation of neighboring proxy-strain differences.
- A positive number prescribes a fixed scale in the internal strain-rate units, $\mathrm{mm\,m^{-1}\,yr^{-1}}$.

A smaller scale makes the graph more sensitive to proxy-strain differences. A larger scale weakens this sensitivity and makes the weighting behave more like distance-only weighting.

#### Morozov discrepancy parameter

- `MOROZOV_TAU = 1.0` controls the target weighted residual, where $\hat{\mathbf e}=\mathbf y-\mathbf A\hat{\mathbf x}$:

$$
\hat{\mathbf e}^{\mathsf{T}}\mathbf{W}\hat{\mathbf e}
=
\tau n_d,
\qquad
n_d = 2n_{\mathrm{RUM}}.
$$

- `1.0` is the manuscript setting.
- Values greater than `1.0` permit a larger residual and generally lead to stronger regularization.
- Values below `1.0` enforce a closer fit and may begin to fit observational noise.

#### Regularization-parameter search interval

- `LOG10_ALPHA_BOUNDS = (-6.0, 3.0)` defines the interval in which the Morozov regularization parameter is searched.
- The default interval corresponds to

$$
10^{-6}\leq\alpha\leq10^3.
$$

- The interval may be widened if the selected value of $\alpha$ lies close to either boundary.

In [4]:
# ============================================================
# --- Inversion settings ---
# ============================================================
# The default values reproduce the Jakarta manuscript result.
# Changing these values produces a different inversion.

RUM_WIDTH_M = 450.0  # [m]; must match the RUM footprint and grid spacing.
GAUSS_ORDER = 3  # Integer >= 3; use 3 for the manuscript result.

STRAIN_GRAPH_RADIUS_M = 3.0 * RUM_WIDTH_M  # [m]; manuscript value: d_thresh = 1350 m.
STRAIN_GRAPH_WEIGHTING = "tensor_sim"  # "tensor_sim" or "distance".
STRAIN_GRAPH_DISTANCE_KERNEL = "quadratic"  # "quadratic", "exponential", or "binary".
STRAIN_GRAPH_VOLUMETRIC_ONLY = True  # True: proxy I1; False: proxy principal strains.
STRAIN_SIMILARITY_SCALE = None  # None: MAD estimate; or a positive fixed scale.

MOROZOV_TAU = 1.0  # Positive value; 1.0 reproduces the manuscript result.
LOG10_ALPHA_BOUNDS = (-6.0, 3.0)  # Search alpha from 1e-6 to 1e3.


In [5]:
if RUN_FULL_INVERSION:
    # Sparse Q8 finite-element inversion of area-average RUM velocities.
    # The method estimates a continuous nodal velocity field using the
    # manuscript observation, weighting, and regularization operators.

    # =================================================
    # --- Memory prompt / purpose of this script ---
    # =================================================
    # This script performs a sparse 2D horizontal FEM inversion of RUM-average
    # InSAR velocity observations. Each 450 m RUM cell is represented by one Q8
    # serendipity rectangular FEM element. The unknown model is the nodal East and
    # North velocity field:
    #
    #     x_hat = [estimated East nodal velocities; estimated North nodal velocities]
    #
    # The observation operator predicts the area-average East/North velocity over
    # each RUM cell, not a point value at the cell center. The inversion solves a
    # weighted Tikhonov problem with full East-North observation covariance:
    #
    #     minimize ||A x_hat - y||_{Q_yy^-1}^2 + alpha^2 ||L x_hat||^2
    #
    # where alpha is chosen by the Morozov discrepancy principle. The regularization
    # combines velocity-gradient smoothing and strain-space graph smoothing between
    # neighboring elements.
    #
    # The code is sparse because the production mesh is too large for dense
    # matrices. It does NOT form the full posterior nodal covariance Qx. Instead it
    # computes exact local covariance blocks needed for interpretation:
    #
    #     Q_y_hat_rum[i] = 2x2 covariance of [E_pred_i, N_pred_i]
    #     Q_eps_hat_elem[e]   = 3x3 covariance of [exx_e, eyy_e, exy_e]
    #
    # These local strain covariance blocks are sufficient to propagate uncertainty
    # into strain-rate quantities such as I1_hat and maximum gamma_max_hat using
    # the delta method.

    import time

    import numpy as np
    import matplotlib.pyplot as plt
    import scipy.sparse as sp
    import scipy.sparse.linalg as spla
    from scipy.optimize import minimize_scalar
    from scipy.spatial import cKDTree


    start_time = time.time()

    # Full Qx is not formed. For your production-size mesh it would be about
    # 40562 x 40562, which is too large. Instead, the code computes exact local
    # covariance blocks for the quantities you need:
    #   Q_eps_hat_elem[e] = 3 x 3 covariance of [eps_EE_hat, eps_NN_hat, eps_EN_hat] for element e.
    LOCAL_COVARIANCE_BATCH_SIZE = 64


    # =================================================
    # --- Required input arrays expected in namespace ---
    # =================================================
    # The script assumes these arrays already exist in the Python/Jupyter namespace.
    # They should describe the RUM cell centers, horizontal velocities, and the
    # measurement covariance for East and North velocity components.
    # E_rum_input, N_rum_input
    # y_E_input, y_N_input
    # sigma_y_E_input, sigma_y_N_input
    # cov_y_EN_input


    # =================================================
    # --- Prepare the complete RUM input arrays ---
    # =================================================
    E_rum = np.asarray(E_rum_input, dtype=float)
    N_rum = np.asarray(N_rum_input, dtype=float)

    y_E = np.asarray(y_E_input, dtype=float)
    y_N = np.asarray(y_N_input, dtype=float)

    sigma_y_E = np.asarray(sigma_y_E_input, dtype=float)
    sigma_y_N = np.asarray(sigma_y_N_input, dtype=float)
    cov_y_EN = np.asarray(cov_y_EN_input, dtype=float)

    if not (
        len(E_rum) == len(N_rum) == len(y_E) == len(y_N)
        == len(sigma_y_E) == len(sigma_y_N) == len(cov_y_EN)
    ):
        raise ValueError("Input RUM coordinate, velocity, sigma, and covariance arrays must have equal length.")

    if np.any(sigma_y_E <= 0) or np.any(sigma_y_N <= 0):
        raise ValueError("All East and North standard deviations must be positive.")


    # =================================================
    # --- FEM cell size ---
    # =================================================
    # RUM cells and FEM elements are assumed to have the same rectangular footprint.
    rum_width = RUM_WIDTH_M
    half_rum_width = rum_width / 2.0


    # =================================================
    # --- Step 1: Build continuous FEM grid ---
    # =================================================
    # Build a regular rectangular grid whose element centers align with the RUM
    # cells. The outer half-cell padding makes each RUM center fall inside one
    # 450 m by 450 m FEM element.
    x_min, x_max = np.min(E_rum) - half_rum_width, np.max(E_rum) + half_rum_width
    y_min, y_max = np.min(N_rum) - half_rum_width, np.max(N_rum) + half_rum_width

    x_nodes = np.arange(x_min, x_max + rum_width, rum_width)
    y_nodes = np.arange(y_min, y_max + rum_width, rum_width)

    xx, yy = np.meshgrid(x_nodes, y_nodes)
    node_coords = np.column_stack([xx.ravel(), yy.ravel()])

    n_x = len(x_nodes)
    n_y = len(y_nodes)


    # =================================================
    # --- Step 1b: Define FEM elements (Q8 serendipity rectangles) ---
    # =================================================
    # Each element has 8 nodes: 4 corners and 4 mid-edge nodes, with no center node.
    # Mid-edge nodes are shared between neighboring elements through mid_nodes_dict.
    elements = []
    node_coords_list = node_coords.tolist()
    mid_nodes_dict = {}
    current_node_idx = len(node_coords_list)

    for j in range(n_y - 1):
        for i in range(n_x - 1):
            n1 = j * n_x + i
            n2 = n1 + 1
            n4 = n1 + n_x
            n3 = n4 + 1

            edges = [(n1, n2), (n2, n3), (n3, n4), (n4, n1)]
            mid_nodes = []

            for edge in edges:
                key = tuple(sorted(edge))
                if key in mid_nodes_dict:
                    mid_nodes.append(mid_nodes_dict[key])
                else:
                    x_mid = 0.5 * (node_coords_list[edge[0]][0] + node_coords_list[edge[1]][0])
                    y_mid = 0.5 * (node_coords_list[edge[0]][1] + node_coords_list[edge[1]][1])
                    node_coords_list.append([x_mid, y_mid])
                    mid_nodes_dict[key] = current_node_idx
                    mid_nodes.append(current_node_idx)
                    current_node_idx += 1

            n5, n6, n7, n8 = mid_nodes
            elements.append([n1, n2, n3, n4, n5, n6, n7, n8])

    elements = np.asarray(elements, dtype=int)
    node_coords = np.asarray(node_coords_list, dtype=float)
    n_nodes = len(node_coords)
    n_elem = len(elements)

    print(f"Number of FEM nodes (Q8): {n_nodes}")
    print(f"Number of FEM elements: {n_elem}")


    # =================================================
    # --- Step 2: Q8 shape functions and derivatives ---
    # =================================================
    # These functions define the interpolation basis and its derivatives on the
    # reference square xi, eta in [-1, 1]. They are used for both cell averaging and
    # strain/regularization operators.
    def shape_functions_Q8(xi, eta):
        N = np.zeros(8)
        N[0] = 0.25 * (1 - xi) * (1 - eta) * (-xi - eta - 1)
        N[1] = 0.25 * (1 + xi) * (1 - eta) * (xi - eta - 1)
        N[2] = 0.25 * (1 + xi) * (1 + eta) * (xi + eta - 1)
        N[3] = 0.25 * (1 - xi) * (1 + eta) * (-xi + eta - 1)
        N[4] = 0.5 * (1 - xi**2) * (1 - eta)
        N[5] = 0.5 * (1 + xi) * (1 - eta**2)
        N[6] = 0.5 * (1 - xi**2) * (1 + eta)
        N[7] = 0.5 * (1 - xi) * (1 - eta**2)
        return N


    def dN_dxi_eta_Q8(xi, eta):
        dN_dxi = np.array([
            0.25 * (1 - eta) * (2 * xi + eta),
            0.25 * (1 - eta) * (2 * xi - eta),
            0.25 * (1 + eta) * (2 * xi + eta),
            0.25 * (1 + eta) * (2 * xi - eta),
            -xi * (1 - eta),
            0.5 * (1 - eta**2),
            -xi * (1 + eta),
            -0.5 * (1 - eta**2),
        ])

        dN_deta = np.array([
            0.25 * (1 - xi) * (xi + 2 * eta),
            -0.25 * (1 + xi) * (xi - 2 * eta),
            0.25 * (1 + xi) * (xi + 2 * eta),
            -0.25 * (1 - xi) * (xi - 2 * eta),
            -0.5 * (1 - xi**2),
            -eta * (1 + xi),
            0.5 * (1 - xi**2),
            -eta * (1 - xi),
        ])

        return dN_dxi, dN_deta


    def element_jacobian_rect(elem, node_coords):
        xe = node_coords[elem, 0]
        ye = node_coords[elem, 1]
        dx = xe[1] - xe[0]
        dy = ye[3] - ye[0]

        if dx <= 0 or dy <= 0:
            raise ValueError("Elements must be axis-aligned rectangles with positive dx and dy.")

        J = np.array([[dx / 2.0, 0.0], [0.0, dy / 2.0]])
        return J, np.linalg.inv(J), abs(np.linalg.det(J))


    def sparse_fro_norm(M):
        return float(np.sqrt(M.multiply(M).sum()))


    def sparse_row_abs_mean(M):
        return float(np.mean(np.asarray(abs(M).sum(axis=1)).ravel()))


    # =================================================
    # --- Step 3: Element lookup for each RUM cell ---
    # =================================================
    # For every RUM center, find the FEM element that contains it. This creates the
    # mapping from observation row to host element.
    ix = np.floor((E_rum - x_min) / rum_width).astype(int)
    iy = np.floor((N_rum - y_min) / rum_width).astype(int)

    if np.any(ix < 0) or np.any(ix >= n_x - 1) or np.any(iy < 0) or np.any(iy >= n_y - 1):
        raise ValueError("Some RUMs are outside the FEM grid.")

    rum_to_fe = iy * (n_x - 1) + ix


    # =================================================
    # --- Step 4: Sparse element-average observation matrix ---
    # =================================================
    # Build A so that A @ nodal_velocity gives the area-average velocity over each
    # host element. A is sparse because each RUM row touches only the 8 Q8 nodes of
    # its own element.
    def build_A_element_average_sparse(elements, node_coords, rum_to_fe, n_gp_side=GAUSS_ORDER):
        n_rum = len(rum_to_fe)
        n_nodes = node_coords.shape[0]

        gp_1d, w_1d = np.polynomial.legendre.leggauss(n_gp_side)
        elem_weights = {}

        for e in np.unique(rum_to_fe):
            elem = elements[e]
            _, _, detJ = element_jacobian_rect(elem, node_coords)

            wsum = np.zeros(8)
            area = 0.0

            for a, xi in enumerate(gp_1d):
                for b, eta in enumerate(gp_1d):
                    weight = w_1d[a] * w_1d[b] * detJ
                    wsum += shape_functions_Q8(xi, eta) * weight
                    area += weight

            elem_weights[e] = wsum / area

        rows, cols, data = [], [], []
        for k, e in enumerate(rum_to_fe):
            rows.extend([k] * 8)
            cols.extend(elements[e])
            data.extend(elem_weights[e])

        return sp.csr_matrix((data, (rows, cols)), shape=(n_rum, n_nodes))


    A_component = build_A_element_average_sparse(elements, node_coords, rum_to_fe, n_gp_side=GAUSS_ORDER)
    print("Sparse one-component observation matrix built successfully.")
    print(f"A_component shape: {A_component.shape}, nnz: {A_component.nnz}")

    n_rum = A_component.shape[0]

    A = sp.block_diag((A_component, A_component), format="csr")
    y = np.hstack([y_E, y_N])


    # =================================================
    # --- Sparse covariance / weight matrix in [E_all, N_all] order ---
    # =================================================
    # Build W = Q_yy^-1 for the stacked observation vector y = [E_all, N_all]. Each RUM has
    # a 2x2 East/North covariance block, including cov_y_EN_input.
    def build_weight_matrix_EN_sparse(sigE, sigN, covEN):
        n = len(sigE)
        rows, cols, data = [], [], []

        for i in range(n):
            C_i = np.array([
                [sigE[i] ** 2, covEN[i]],
                [covEN[i], sigN[i] ** 2],
            ])

            if np.linalg.det(C_i) <= 0:
                raise ValueError(f"Non-positive-definite EN covariance at RUM index {i}.")

            W_i = np.linalg.inv(C_i)
            inds = [(i, i), (i, n + i), (n + i, i), (n + i, n + i)]
            vals = [W_i[0, 0], W_i[0, 1], W_i[1, 0], W_i[1, 1]]

            for (r, c), v in zip(inds, vals):
                rows.append(r)
                cols.append(c)
                data.append(v)

        return sp.csr_matrix((data, (rows, cols)), shape=(2 * n, 2 * n))


    W = build_weight_matrix_EN_sparse(sigma_y_E, sigma_y_N, cov_y_EN)


    # =================================================
    # --- Optional mesh plot ---
    # =================================================
    # Quick visual check of the first few FEM elements and the RUM centers. This is
    # not part of the inversion; it helps catch coordinate/grid mistakes early.
    plt.figure(figsize=(8, 8))
    for elem in elements[: min(50, len(elements))]:
        n1, n2, n3, n4, n5, n6, n7, n8 = elem
        edges = [[n1, n5, n2], [n2, n6, n3], [n3, n7, n4], [n4, n8, n1]]
        for edge in edges:
            plt.plot(node_coords[edge, 0], node_coords[edge, 1], "k-", linewidth=0.8)
    plt.scatter(node_coords[:, 0], node_coords[:, 1], c="k", s=10, label="FEM nodes")
    plt.scatter(E_rum, N_rum, c="r", s=20, label="RUM centers", alpha=0.6)
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.show()


    # =================================================
    # --- Sparse velocity first-gradient smoothing matrix ---
    # =================================================
    # Assemble the scalar velocity-gradient smoothness matrix for one component.
    # Later it is duplicated for East and North with a block diagonal matrix.
    def build_velocity_gradient_smoothing_Q8_sparse(elements, node_coords, n_gp_side=GAUSS_ORDER):
        n_nodes = node_coords.shape[0]
        gp_1d, w_1d = np.polynomial.legendre.leggauss(n_gp_side)

        rows, cols, data = [], [], []

        for elem in elements:
            _, invJ, detJ = element_jacobian_rect(elem, node_coords)
            K_v_e = np.zeros((8, 8))

            for a, xi in enumerate(gp_1d):
                for b, eta in enumerate(gp_1d):
                    dN_dxi, dN_deta = dN_dxi_eta_Q8(xi, eta)
                    dN_dxdy = invJ @ np.vstack((dN_dxi, dN_deta))

                    dN_dx = dN_dxdy[0, :]
                    dN_dy = dN_dxdy[1, :]

                    G_e = np.vstack([
                        dN_dx,
                        dN_dy,
                        dN_dx,
                        dN_dy,
                        0.5 * (dN_dx + dN_dy),
                        0.5 * (dN_dx + dN_dy),
                    ])

                    weight = w_1d[a] * w_1d[b] * detJ
                    K_v_e += G_e.T @ G_e * weight

            rr, cc = np.meshgrid(elem, elem, indexing="ij")
            rows.extend(rr.ravel())
            cols.extend(cc.ravel())
            data.extend(K_v_e.ravel())

        K_v = sp.coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes)).tocsr()
        row_sum = sparse_row_abs_mean(K_v)
        if row_sum > 0:
            K_v = K_v / row_sum

        return K_v


    K_v = build_velocity_gradient_smoothing_Q8_sparse(elements, node_coords, n_gp_side=GAUSS_ORDER)
    print(f"Velocity-gradient smoothing matrix shape: {K_v.shape}, nnz: {K_v.nnz}")


    # =================================================
    # --- Sparse area-average strain operator ---
    # =================================================
    # Build B so that B @ x_hat gives [eps_EE_hat, eps_NN_hat, eps_EN_hat] per element, averaged
    # over the full element area. This is the strain operator used for reporting and
    # for local strain covariance propagation.
    def build_area_average_strain_operator_Q8_sparse(elements, node_coords, n_gp_side=GAUSS_ORDER):
        n_elem = len(elements)
        n_nodes = node_coords.shape[0]
        gp_1d, w_1d = np.polynomial.legendre.leggauss(n_gp_side)

        rows, cols, data = [], [], []

        for e, elem in enumerate(elements):
            _, invJ, detJ = element_jacobian_rect(elem, node_coords)

            avg_dN_dx = np.zeros(8)
            avg_dN_dy = np.zeros(8)
            area = 0.0

            for a, xi in enumerate(gp_1d):
                for b, eta in enumerate(gp_1d):
                    dN_dxi, dN_deta = dN_dxi_eta_Q8(xi, eta)
                    dN_dxdy = invJ @ np.vstack((dN_dxi, dN_deta))

                    weight = w_1d[a] * w_1d[b] * detJ
                    avg_dN_dx += dN_dxdy[0, :] * weight
                    avg_dN_dy += dN_dxdy[1, :] * weight
                    area += weight

            avg_dN_dx /= area
            avg_dN_dy /= area

            for local, node in enumerate(elem):
                rows.append(3 * e + 0)
                cols.append(node)
                data.append(avg_dN_dx[local])

                rows.append(3 * e + 1)
                cols.append(n_nodes + node)
                data.append(avg_dN_dy[local])

                rows.append(3 * e + 2)
                cols.append(node)
                data.append(0.5 * avg_dN_dy[local])

                rows.append(3 * e + 2)
                cols.append(n_nodes + node)
                data.append(0.5 * avg_dN_dx[local])

        return sp.csr_matrix((data, (rows, cols)), shape=(3 * n_elem, 2 * n_nodes))


    B = build_area_average_strain_operator_Q8_sparse(elements, node_coords, n_gp_side=GAUSS_ORDER)
    print(f"Area-average strain operator shape: {B.shape}, nnz: {B.nnz}")


    # =================================================
    # --- Sparse element strain-space Laplacian smoothing ---
    # =================================================
    # Build a graph Laplacian in strain space. Neighboring elements are coupled by
    # distance and, optionally, by a tensor-similarity proxy computed from local
    # affine fits to the observed RUM velocities.
    def build_L_strain_sparse(
        elements,
        node_coords,
        d_thresh,
        rum_coords=None,
        rum_vel_e=None,
        rum_vel_n=None,
        smoothing_type=STRAIN_GRAPH_DISTANCE_KERNEL,
        weighting_mode=STRAIN_GRAPH_WEIGHTING,
        volumetric_only=STRAIN_GRAPH_VOLUMETRIC_ONLY,
        sigma_I1_proxy=STRAIN_SIMILARITY_SCALE,
    ):
        n_elem = len(elements)
        centroids = np.array([node_coords[elem].mean(axis=0) for elem in elements])

        tree = cKDTree(centroids)
        neighbor_pairs = np.asarray(list(tree.query_pairs(d_thresh)), dtype=int)

        if neighbor_pairs.size == 0:
            return sp.csr_matrix((3 * n_elem, 3 * n_elem))

        if weighting_mode == "tensor_sim":
            if rum_coords is None or rum_vel_e is None or rum_vel_n is None:
                raise ValueError("tensor_sim requires RUM coordinates and horizontal velocity data.")

            rum_tree = cKDTree(rum_coords)
            eps_proxy = np.full((n_elem, 2, 2), np.nan)

            for e, center in enumerate(centroids):
                idx = rum_tree.query_ball_point(center, d_thresh)
                if len(idx) < 3:
                    continue

                idx = np.asarray(idx, dtype=int)
                x = rum_coords[idx, 0]
                y = rum_coords[idx, 1]
                u = rum_vel_e[idx]
                v = rum_vel_n[idx]

                X = np.column_stack([np.ones_like(x), x, y])
                au, *_ = np.linalg.lstsq(X, u, rcond=None)
                av, *_ = np.linalg.lstsq(X, v, rcond=None)

                grad_v_proxy = np.array([[au[1], au[2]], [av[1], av[2]]])
                eps_proxy[e] = 0.5 * (grad_v_proxy + grad_v_proxy.T)

            if sigma_I1_proxy is None:
                dEs = []
                for i, j in neighbor_pairs:
                    if np.isnan(eps_proxy[i, 0, 0]) or np.isnan(eps_proxy[j, 0, 0]):
                        continue

                    if volumetric_only:
                        trace_i = np.trace(eps_proxy[i])
                        trace_j = np.trace(eps_proxy[j])
                        if trace_i * trace_j < 0:
                            continue
                        delta_I1 = abs(trace_i - trace_j)
                    else:
                        eig_i = np.linalg.eigvalsh(eps_proxy[i])
                        eig_j = np.linalg.eigvalsh(eps_proxy[j])
                        delta_I1 = np.linalg.norm(eig_i - eig_j)

                    dEs.append(delta_I1)

                if len(dEs) == 0:
                    sigma_I1_proxy = 1.0
                    print("Warning: no valid strain contrasts found; using strain_scale = 1.0.")
                else:
                    dEs = np.asarray(dEs)
                    mad = 1.4826 * np.median(np.abs(dEs - np.median(dEs)))
                    sigma_I1_proxy = max(mad, np.finfo(float).eps)
        elif weighting_mode != "distance":
            raise ValueError(f"Unknown weighting_mode: {weighting_mode}")

        rows, cols, data = [], [], []

        for i, j in neighbor_pairs:
            d = np.linalg.norm(centroids[i] - centroids[j])

            if smoothing_type == "quadratic":
                w_dist = 1.0 / (1.0 + (d / d_thresh) ** 2)
            elif smoothing_type == "exponential":
                w_dist = np.exp(-(d / d_thresh) ** 2)
            elif smoothing_type == "binary":
                w_dist = 1.0
            else:
                raise ValueError(f"Unknown smoothing_type: {smoothing_type}")

            if weighting_mode == "distance":
                w_ek = w_dist
            else:
                if np.isnan(eps_proxy[i, 0, 0]) or np.isnan(eps_proxy[j, 0, 0]):
                    w_ek = 0.0
                else:
                    if volumetric_only:
                        trace_i = np.trace(eps_proxy[i])
                        trace_j = np.trace(eps_proxy[j])
                        if trace_i * trace_j < 0:
                            w_ek = 0.1 * w_dist
                        else:
                            delta_I1 = abs(trace_i - trace_j)
                            w_I1 = np.exp(-(delta_I1 / sigma_I1_proxy) ** 2)
                            w_min = min(0.3, w_dist)
                            w_ek = w_dist * max(w_I1, w_min)
                    else:
                        eig_i = np.linalg.eigvalsh(eps_proxy[i])
                        eig_j = np.linalg.eigvalsh(eps_proxy[j])
                        delta_I1 = np.linalg.norm(eig_i - eig_j)
                        w_I1 = np.exp(-(delta_I1 / sigma_I1_proxy) ** 2)
                        w_min = min(0.3, w_dist)
                        w_ek = w_dist * max(w_I1, w_min)

            for k in range(3):
                ii = 3 * i + k
                jj = 3 * j + k
                rows.extend([ii, jj, ii, jj])
                cols.extend([ii, jj, jj, ii])
                data.extend([w_ek, w_ek, -w_ek, -w_ek])

        Ls = sp.coo_matrix((data, (rows, cols)), shape=(3 * n_elem, 3 * n_elem)).tocsr()
        row_sum = sparse_row_abs_mean(Ls)
        if row_sum > 0:
            Ls = Ls / row_sum

        return Ls


    rum_coords_array = np.column_stack([E_rum, N_rum])

    L_eps = build_L_strain_sparse(
        elements,
        node_coords=node_coords,
        rum_coords=rum_coords_array,
        rum_vel_e=y_E,
        rum_vel_n=y_N,
        d_thresh=STRAIN_GRAPH_RADIUS_M,
        weighting_mode=STRAIN_GRAPH_WEIGHTING,
        smoothing_type=STRAIN_GRAPH_DISTANCE_KERNEL,
        volumetric_only=STRAIN_GRAPH_VOLUMETRIC_ONLY,
        sigma_I1_proxy=STRAIN_SIMILARITY_SCALE,
    )

    print(f"Strain-space graph Laplacian shape: {L_eps.shape}, nnz: {L_eps.nnz}")


    # =================================================
    # --- Total 2D regularization ---
    # =================================================
    # Combine the two regularization pieces into one sparse operator:
    #   1. velocity smoothness for East and North nodal velocities
    #   2. strain-space graph smoothing applied to area-average strain
    # The strain rows are normalized component-wise so eps_EE_hat, eps_NN_hat, and eps_EN_hat contribute
    # on comparable numerical scales.
    L_v = sp.block_diag((K_v, K_v), format="csr")
    L_eps_to_x = (L_eps @ B).tocsr()

    scale_v = sparse_fro_norm(L_v)
    L_eps_to_x_normalized = L_eps_to_x.tolil(copy=True)

    components = ["eps_EE", "eps_NN", "eps_EN"]
    norm_blocks = {}
    eps_floor = 1e-12

    for i, comp in enumerate(components):
        rows = np.arange(i, 3 * n_elem, 3)
        sub_block = L_eps_to_x[rows, :]
        norm_block = sparse_fro_norm(sub_block)
        norm_blocks[comp] = norm_block
        factor = scale_v / max(norm_block, eps_floor)
        L_eps_to_x_normalized[rows, :] = L_eps_to_x_normalized[rows, :] * factor

    L_eps_to_x_normalized = L_eps_to_x_normalized.tocsr()

    L = sp.vstack([L_v, L_eps_to_x_normalized], format="csr")

    print(f"Total regularization shape: {L.shape}, nnz: {L.nnz}")
    print("Original strain-graph block norms:", norm_blocks)


    # =================================================
    # --- Sparse Tikhonov solver ---
    # =================================================
    # Solve the regularized normal equations:
    #   (A^T W A + alpha^2 L^T L) V = A^T W d
    # using sparse linear algebra. The helper functions also build/factor the final
    # matrix so it can be reused for exact local covariance calculations.
    def solve_tikhonov_sparse(
        ATWA,
        LTL,
        ATWy,
        A,
        L,
        y,
        W,
        alpha,
        compute_cov=False,
        eps_rel=1e-12,
    ):
        M = build_system_matrix(ATWA, LTL, alpha, eps_rel=eps_rel)

        x_hat = spla.spsolve(M, ATWy)
        e_hat = y - A @ x_hat
        W_e_hat = W @ e_hat
        e_norm = np.sqrt(float(e_hat.T @ W_e_hat))
        s_norm = np.linalg.norm(L @ x_hat)

        return x_hat, e_norm, s_norm, None, None


    def build_system_matrix(ATWA, LTL, alpha, eps_rel=1e-12):
        M = (ATWA + alpha**2 * LTL).tocsc()
        eps = eps_rel * M.diagonal().mean()
        if eps <= 0:
            eps = eps_rel
        return M + sp.eye(M.shape[0], format="csc") * eps


    def factorize_system(ATWA, LTL, alpha, eps_rel=1e-12):
        M = build_system_matrix(ATWA, LTL, alpha, eps_rel=eps_rel)
        return spla.splu(M)


    def exact_local_covariance_blocks(factor, ATWA, G, n_blocks, block_size, batch_size=64):
        """
        Exact covariance blocks for y = G x without forming full Qx.

        The estimator covariance is:
            Qx = M^-1 ATA M^-1

        For each local block G_e:
            Q_y,e = G_e Qx G_e^T
                  = Z_e^T ATA Z_e
            where M Z_e = G_e^T.

        This returns only the diagonal local blocks, e.g. one 3x3 strain covariance
        matrix per element. It is exact for those blocks.
        """
        Q_blocks = np.zeros((n_blocks, block_size, block_size))

        for start in range(0, n_blocks, batch_size):
            stop = min(start + batch_size, n_blocks)
            rows = np.arange(block_size * start, block_size * stop)

            rhs_block = G[rows, :].T.toarray()
            Z = factor.solve(rhs_block)
            AZ = ATWA @ Z
            Q_batch = Z.T @ AZ

            for local_block, global_block in enumerate(range(start, stop)):
                a = block_size * local_block
                b = a + block_size
                Qe = Q_batch[a:b, a:b]
                Q_blocks[global_block] = 0.5 * (Qe + Qe.T)

            print(f"Computed local covariance blocks {start + 1}-{stop} / {n_blocks}")

        return Q_blocks


    def exact_interleaved_prediction_covariance_blocks(factor, ATWA, A, n_rum, batch_size=128):
        """
        Exact 2x2 covariance block for each predicted RUM vector [E_i, N_i].

        Returns Q_y_hat_rum[i] =
            [[var(E_i), cov(E_i,N_i)],
             [cov(N_i,E_i), var(N_i)]]

        It avoids forming the full 2*n_rum by 2*n_rum Qy matrix.
        """
        Q_blocks = np.zeros((n_rum, 2, 2))

        for start in range(0, n_rum, batch_size):
            stop = min(start + batch_size, n_rum)

            rows = []
            for i in range(start, stop):
                rows.extend([i, n_rum + i])
            rows = np.asarray(rows, dtype=int)

            rhs_block = A[rows, :].T.toarray()
            Z = factor.solve(rhs_block)
            AZ = ATWA @ Z
            Q_batch = Z.T @ AZ

            for local_block, global_block in enumerate(range(start, stop)):
                a = 2 * local_block
                b = a + 2
                Qe = Q_batch[a:b, a:b]
                Q_blocks[global_block] = 0.5 * (Qe + Qe.T)

            print(f"Computed RUM prediction covariance blocks {start + 1}-{stop} / {n_rum}")

        return Q_blocks


    # =================================================
    # --- Select alpha by Morozov discrepancy principle ---
    # =================================================
    # Pick alpha so that the weighted residual is close to the expected noise level.
    # With y = [E_all, N_all], len(d) = 2*n_rum, so tau=1 targets roughly one
    # weighted residual unit per data component when the covariance is realistic.
    def select_alpha_sparse(
        ATWA,
        LTL,
        ATWy,
        A,
        L,
        y,
        W,
        tau=MOROZOV_TAU,
        log_alpha_bounds=LOG10_ALPHA_BOUNDS,
    ):
        target = tau * len(y)

        def objective(log_alpha):
            alpha = 10**log_alpha
            x_hat_objective, _, _, _, _ = solve_tikhonov_sparse(
                ATWA, LTL, ATWy, A, L, y, W, alpha, compute_cov=False
            )
            e_hat = y - A @ x_hat_objective
            return abs(float(e_hat.T @ (W @ e_hat)) - target)

        res = minimize_scalar(objective, bounds=log_alpha_bounds, method="bounded")
        alpha = 10**res.x

        x_hat_opt, e_norm, s_norm, _, _ = solve_tikhonov_sparse(
            ATWA, LTL, ATWy, A, L, y, W, alpha, compute_cov=False
        )

        weighted_residual = e_norm**2
        print(
            f"Discrepancy principle: weighted residual = {weighted_residual:.3f}, "
            f"target = {target:.3f} (tau={tau})"
        )

        return alpha, x_hat_opt


    # =================================================
    # --- Precompute and solve full 2D system ---
    # =================================================
    # Precompute the sparse normal-equation pieces once. Alpha selection then reuses
    # them repeatedly instead of rebuilding A^T W A or L^T L every time.
    ATWA = (A.T @ W @ A).tocsr()
    ATWy = A.T @ (W @ y)
    LTL = (L.T @ L).tocsr()

    print(f"Normal equations precomputed. ATWA nnz={ATWA.nnz}, LTL nnz={LTL.nnz}")

    alpha, x_hat = select_alpha_sparse(
        ATWA,
        LTL,
        ATWy,
        A,
        L,
        y,
        W,
        tau=MOROZOV_TAU,
        log_alpha_bounds=LOG10_ALPHA_BOUNDS,
    )

    print(f"Optimal alpha = {alpha:.2e}")

    # Factor the final system once. This factorization is reused for the exact
    # local covariance blocks below.
    print("Factoring final sparse system for solution and local covariance...")
    M_factor = factorize_system(ATWA, LTL, alpha)
    x_hat = M_factor.solve(ATWy)


    # =================================================
    # --- Extract velocity field, predictions, residuals ---
    # =================================================
    # Split the solved vector back into East/North nodal velocities and compare the
    # FEM-predicted RUM area averages against the observed RUM velocities.
    v_E_nodes_hat = x_hat[:n_nodes]
    v_N_nodes_hat = x_hat[n_nodes:]

    y_hat = A @ x_hat
    y_E_hat = y_hat[:n_rum]
    y_N_hat = y_hat[n_rum:]

    e_E_hat = y_E - y_E_hat
    e_N_hat = y_N - y_N_hat


    # =================================================
    # --- Exact local RUM prediction covariance blocks ---
    # =================================================
    # Compute only the 2x2 covariance block for each predicted RUM vector
    # [E_pred_i, N_pred_i]. This is exact, but avoids forming the full dense Qy.
    print("Computing exact 2x2 prediction covariance block for each RUM...")
    Q_y_hat_rum = exact_interleaved_prediction_covariance_blocks(
        factor=M_factor,
        ATWA=ATWA,
        A=A,
        n_rum=n_rum,
        batch_size=128,
    )

    sigma_y_E_hat = np.sqrt(np.maximum(Q_y_hat_rum[:, 0, 0], 0.0))
    sigma_y_N_hat = np.sqrt(np.maximum(Q_y_hat_rum[:, 1, 1], 0.0))
    cov_y_EN_hat = Q_y_hat_rum[:, 0, 1]


    # =================================================
    # --- Compute compatible area-average strain tensors ---
    # =================================================
    # Convert the solved nodal velocity field into one area-average strain tensor
    # per FEM element.
    eps_hat_components = np.asarray(B @ x_hat).reshape(-1, 3)

    eps_EE_hat = eps_hat_components[:, 0]
    eps_NN_hat = eps_hat_components[:, 1]
    eps_EN_hat = eps_hat_components[:, 2]

    print(f"Computed area-average strain tensors for {eps_hat_components.shape[0]} FEM elements.")


    # =================================================
    # --- Exact local strain covariance blocks ---
    # =================================================
    # Compute Q_eps_hat_elem[e], the exact 3x3 covariance block of
    # [exx_e, eyy_e, exy_e] for each element. This is the covariance object needed
    # for uncertainty propagation into nonlinear strain invariants.
    print("Computing exact 3x3 strain covariance block for each element...")
    Q_eps_hat_elem = exact_local_covariance_blocks(
        factor=M_factor,
        ATWA=ATWA,
        G=B,
        n_blocks=n_elem,
        block_size=3,
        batch_size=LOCAL_COVARIANCE_BATCH_SIZE,
    )

    sigma_eps_EE = np.sqrt(np.maximum(Q_eps_hat_elem[:, 0, 0], 0.0))
    sigma_eps_NN = np.sqrt(np.maximum(Q_eps_hat_elem[:, 1, 1], 0.0))
    sigma_eps_EN = np.sqrt(np.maximum(Q_eps_hat_elem[:, 2, 2], 0.0))

    print("Finished exact local strain covariance blocks.")


    # =================================================
    # --- Plot helpers ---
    # =================================================
    # Small plotting utilities for stable color limits. They only affect display,
    # not the inversion or uncertainty calculations.
    def symmetric_color_limit(*arrays, fallback=1.0):
        values = np.concatenate([np.ravel(np.asarray(a, dtype=float)) for a in arrays])
        values = values[np.isfinite(values)]
        if values.size == 0:
            return fallback
        vmax = np.nanmax(np.abs(values))
        return vmax if vmax > 0 else fallback


    def robust_symmetric_color_limit(data, nsigma=2.0, fallback=1.0):
        values = np.ravel(np.asarray(data, dtype=float))
        values = values[np.isfinite(values)]
        if values.size == 0:
            return fallback
        vmax = nsigma * np.nanstd(values)
        if vmax <= 0:
            vmax = np.nanmax(np.abs(values))
        return vmax if vmax > 0 else fallback


    def plot_lcurve_component_sparse(ATWA, LTL, ATWy, A, L, y, W, alpha, component_name="2D horizontal"):
        alphas = np.logspace(-4, 6, 100)
        R, S = [], []

        for alpha_grid in alphas:
            _, e_norm, s_norm, _, _ = solve_tikhonov_sparse(
                ATWA, LTL, ATWy, A, L, y, W, alpha_grid, compute_cov=False
            )
            R.append(e_norm)
            S.append(s_norm)

        _, e_opt, s_opt, _, _ = solve_tikhonov_sparse(
            ATWA, LTL, ATWy, A, L, y, W, alpha, compute_cov=False
        )

        plt.figure(figsize=(6, 6))
        plt.loglog(R, S, "-o", markersize=3, label="L-curve")
        plt.loglog(e_opt, s_opt, "ro", label=f"alpha = {alpha:.2e}")
        plt.xlabel("Weighted residual norm ||C^(-1/2)(Ax-d)||")
        plt.ylabel("Regularization norm ||Lx||")
        plt.title(f"L-curve ({component_name})")
        plt.grid(True, which="both")
        plt.legend()
        plt.tight_layout()
        plt.show()


    # Uncomment to inspect the L-curve. It requires many sparse solves.
    # plot_lcurve_component_sparse(ATWA, LTL, ATWy, A, L, y, W, alpha)


    # =================================================
    # --- Plot RUM observations, FEM predictions, residuals ---
    # =================================================
    # Visual diagnostic for data fit. Residual maps should not show strong spatial
    # patterns if the model, covariance, and regularization are appropriate.
    fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True, sharey=True)

    vmax_E = symmetric_color_limit(y_E, y_E_hat)
    vmax_N = symmetric_color_limit(y_N, y_N_hat)
    vres_E = symmetric_color_limit(e_E_hat)
    vres_N = symmetric_color_limit(e_N_hat)

    for ax, data, title, vmax in zip(
        axes[0],
        [y_E, y_N],
        ["RUM East", "RUM North"],
        [vmax_E, vmax_N],
    ):
        sc = ax.scatter(E_rum, N_rum, c=data, s=120, marker="s",
                        cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(title)
        plt.colorbar(sc, ax=ax)

    for ax, data, title, vmax in zip(
        axes[1],
        [y_E_hat, y_N_hat],
        ["FEM -> RUM East", "FEM -> RUM North"],
        [vmax_E, vmax_N],
    ):
        sc = ax.scatter(E_rum, N_rum, c=data, s=120, marker="s",
                        cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(title)
        plt.colorbar(sc, ax=ax)

    for ax, data, title, vmax in zip(
        axes[2],
        [e_E_hat, e_N_hat],
        ["Residual East (RUM - FEM)", "Residual North (RUM - FEM)"],
        [vres_E, vres_N],
    ):
        sc = ax.scatter(E_rum, N_rum, c=data, s=120, marker="s",
                        cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(title)
        plt.colorbar(sc, ax=ax)

    for ax in axes.flat:
        ax.set_aspect("equal")
        ax.set_xlabel("E [m]")

    for row in range(3):
        axes[row, 0].set_ylabel("N [m]")

    plt.suptitle("RUM velocities vs FEM-predicted area-average velocities + residuals", fontsize=14)
    plt.tight_layout()
    plt.show()


    # =================================================
    # --- Plot area-average strain and local uncertainty ---
    # =================================================
    # Plot the strain tensor components and their exact local 1-sigma uncertainties.
    strain_data = [eps_EE_hat, eps_NN_hat, eps_EN_hat]
    strain_labels = [r"$\hat{\dot{\varepsilon}}_{EE}$", r"$\hat{\dot{\varepsilon}}_{NN}$", r"$\hat{\dot{\varepsilon}}_{EN}$"]
    strain_std_data = [sigma_eps_EE, sigma_eps_NN, sigma_eps_EN]

    elem_centers = np.zeros((n_elem, 2))
    for i, elem in enumerate(elements):
        elem_centers[i] = node_coords[elem].mean(axis=0)

    vmax_strain = [robust_symmetric_color_limit(d, nsigma=2.0) for d in strain_data]
    vmax_std = [
        max(3.0 * np.nanstd(d), np.nanmax(d) if np.any(np.isfinite(d)) else 1.0)
        for d in strain_std_data
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)

    for i, (ax, data, label) in enumerate(zip(axes[0], strain_data, strain_labels)):
        sc = ax.scatter(
            elem_centers[:, 0], elem_centers[:, 1],
            c=data, s=8, marker="o", cmap="seismic_r",
            vmin=-vmax_strain[i], vmax=vmax_strain[i]
        )
        ax.set_title(label)
        ax.set_aspect("equal")
        ax.set_xlabel("E [m]")
        plt.colorbar(sc, ax=ax, shrink=0.9)

    for i, (ax, data, label) in enumerate(zip(axes[1], strain_std_data, strain_labels)):
        sc = ax.scatter(
            elem_centers[:, 0], elem_centers[:, 1],
            c=data, s=8, marker="o", cmap="plasma_r",
            vmin=0, vmax=vmax_std[i]
        )
        ax.set_title(r"$1\sigma$ " + label)
        ax.set_aspect("equal")
        ax.set_xlabel("E [m]")
        plt.colorbar(sc, ax=ax, shrink=0.9)

    axes[0, 0].set_ylabel("N [m]")
    axes[1, 0].set_ylabel("N [m]")

    plt.suptitle("Area-average strain tensor components and exact local uncertainties", fontsize=14)
    plt.tight_layout()
    plt.show()


    print(f"Total runtime: {time.time() - start_time:.2f} s")






### Inversion archive

When the full inversion is skipped, all required quantities are loaded from the distributed inversion archive `results/jakarta_strain_inversion_archive.pkl`.

When the full inversion is executed, this cell assembles the same archive structure. The archive contains:

- finite-element mesh and RUM-to-element mapping information;
- observation matrix $\mathbf{A}$;
- element-average strain operator $\mathbf{B}$;
- observation-weight matrix $\mathbf{W}$;
- velocity-gradient and strain-rate regularization matrices $\mathbf{L}_v$ and $\mathbf{L}_{\varepsilon}$, and the combined matrix $\mathbf{L}$;
- normal-equation terms $\mathbf{A}^{\mathsf{T}}\mathbf{W}\mathbf{A}$, $\mathbf{L}^{\mathsf{T}}\mathbf{L}$, and $\mathbf{A}^{\mathsf{T}}\mathbf{W}\mathbf{y}$;
- selected regularization parameters, including $\alpha$ and $\tau$, and other inversion settings;
- estimated nodal horizontal velocities $\hat{\mathbf{x}}$;
- within-RUM prediction covariance blocks $\mathbf{Q}_{\hat{y},i}$ and within-element strain covariance blocks $\mathbf{Q}_{\hat{\varepsilon},e}$, together with the matrices required for covariance propagation;
- element-average strain-rate tensors $\hat{\dot{\boldsymbol{\varepsilon}}}_e$ and their covariance blocks;
- velocity residuals $\hat{\mathbf{e}}_E$ and $\hat{\mathbf{e}}_N$, and inversion diagnostics.

This common structure allows the remaining notebook cells to operate identically whether the inversion is executed or the precomputed results are loaded.


In [4]:
# ============================================================
# --- Assemble an inversion archive ---
# ============================================================

# A newly executed inversion is stored in the same structure as the
# distributed inversion archive, so all following cells use one common route.
if RUN_FULL_INVERSION:
    e_hat = y - A @ x_hat
    weighted_residual = float(
        e_hat.T @ (W @ e_hat)
    )
    target_weighted_residual = MOROZOV_TAU * len(y)

    archive = {
        "metadata": {
            "archive_schema": "jakarta_strain_inversion_v2",
            "description": "Public Jakarta Q8 strain-rate inversion archive",
            "input_dataset_doi": "10.4121/39e7a39c-d305-425f-b238-1f3a4271dda9.v3",
            "input_dataset_csv_md5": INPUT_CSV_MD5,
            "crs_epsg": 23830,
            "velocity_units": "mm/yr",
            "strain_internal_units": "mm/m/yr",
            "strain_product_units": "microstrain/yr",
            "unknown_order": (
                "v_E_nodes_hat",
                "v_N_nodes_hat",
            ),
            "observation_order": (
                "y_E",
                "y_N",
            ),
            "strain_order": ("eps_EE_hat", "eps_NN_hat", "eps_EN_hat"),
            "strain_quantity_order": ("eps_1_hat", "eps_2_hat", "I1_hat", "gamma_max_hat"),
            "full_Q_x_hat_formed": False,
            "full_Q_y_hat_formed": False,
        },
        "mesh": {
            "node_coords": node_coords,
            "elements": elements,
            "elem_centers": node_coords[elements].mean(axis=1),
            "rum_width_m": rum_width,
            "n_x": n_x,
            "n_y": n_y,
            "n_nodes": n_nodes,
            "n_elements": len(elements),
        },
        "mapping": {
            "rum_to_fe": rum_to_fe,
            "E_rum": E_rum,
            "N_rum": N_rum,
        },
        "operators": {
            "A_component": A_component,
            "A": A,
            "B": B,
            "W": W,
        },
        "regularization": {
            "L": L,
            "K_v": K_v,
            "L_v": L_v,
            "L_eps_to_x": L_eps_to_x,
            "L_eps_to_x_normalized": L_eps_to_x_normalized,
            "L_eps": L_eps,
        },
        "normal_equations": {
            "ATWA": ATWA,
            "LTL": LTL,
            "ATWy": ATWy,
        },
        "hyperparameters": {
            "alpha": alpha,
            "tau": MOROZOV_TAU,
            "strain_graph_radius_m": STRAIN_GRAPH_RADIUS_M,
            "strain_graph_radius_in_rum_widths": (
                STRAIN_GRAPH_RADIUS_M / RUM_WIDTH_M
            ),
            "strain_graph_weighting": STRAIN_GRAPH_WEIGHTING,
            "strain_graph_distance_kernel": STRAIN_GRAPH_DISTANCE_KERNEL,
            "strain_graph_volumetric_only": STRAIN_GRAPH_VOLUMETRIC_ONLY,
            "velocity_regularization_fro_norm": scale_v,
            "strain_graph_component_norms": norm_blocks,
        },
        "solution": {
            "x_hat": x_hat,
            "v_E_nodes_hat": v_E_nodes_hat,
            "v_N_nodes_hat": v_N_nodes_hat,
            "y_E_hat": y_E_hat,
            "y_N_hat": y_N_hat,
        },
        "covariance": {
            "Q_y_hat_rum": Q_y_hat_rum,
            "Q_eps_hat_elem": Q_eps_hat_elem,
        },
        "strain": {
            "eps_hat_components": eps_hat_components,
            "eps_EE_hat": eps_EE_hat,
            "eps_NN_hat": eps_NN_hat,
            "eps_EN_hat": eps_EN_hat,
        },
        "diagnostics": {
            "e_E_hat": e_E_hat,
            "e_N_hat": e_N_hat,
            "weighted_residual": weighted_residual,
            "weighted_residual_per_datum": (
                weighted_residual / len(y)
            ),
            "target_weighted_residual": target_weighted_residual,
        },
    }

# Recover the mesh, strain-rate components, and their local 3 x 3
# covariance blocks for the derivation of strain-rate quantities.
mesh = archive["mesh"]
elements = np.asarray(mesh["elements"], dtype=int)
node_coords = np.asarray(mesh["node_coords"], dtype=float)
rum_to_fe = np.asarray(archive["mapping"]["rum_to_fe"], dtype=int)
eps_EE_hat = np.asarray(archive["strain"]["eps_EE_hat"], dtype=float)
eps_NN_hat = np.asarray(archive["strain"]["eps_NN_hat"], dtype=float)
eps_EN_hat = np.asarray(archive["strain"]["eps_EN_hat"], dtype=float)
Q_eps_hat_elem = np.asarray(
    archive["covariance"]["Q_eps_hat_elem"],
    dtype=float,
)

# Marginal 1-sigma uncertainties of strain-rate tensor components are the square roots of the block diagonals.
sigma_eps_EE = np.sqrt(np.maximum(Q_eps_hat_elem[:, 0, 0], 0.0))
sigma_eps_NN = np.sqrt(np.maximum(Q_eps_hat_elem[:, 1, 1], 0.0))
sigma_eps_EN = np.sqrt(np.maximum(Q_eps_hat_elem[:, 2, 2], 0.0))


## Deriving the strain-rate quantities and within-element covariance blocks

The next cell computes the maximum and minimum principal strain rates, $\dot{\varepsilon}_1$ and $\dot{\varepsilon}_2$, the first strain-rate invariant, $I_1$, and the maximum engineering shear strain rate, $\dot{\gamma}_{\max}$. It also propagates the strain-tensor uncertainty to obtain the complete within-element covariance block of these four derived quantities.


In [5]:
# ============================================================
# --- Derive strain-rate quantities and propagate uncertainties ---
# ============================================================

import numpy as np

# Report maximum engineering shear strain rate. Principal strain rates always
# use the tensorial maximum shear strain rate internally.
ENGINEERING_SHEAR = True


# ============================================================
# --- Helper functions ---
# ============================================================

def _gamma_max_label(engineering_shear):
    if engineering_shear:
        return r"$\hat{\dot{\gamma}}_{\max}$"
    else:
        return r"$\hat{\dot{\gamma}}_{\max}/2$"


def _gamma_max_convention(engineering_shear):
    if engineering_shear:
        return "engineering"
    else:
        return "tensorial"


def _compute_derived_values(
    eps_EE_k,
    eps_NN_k,
    eps_EN_k,
    engineering_shear=False,
):
    tr = eps_EE_k + eps_NN_k
    diff = eps_EE_k - eps_NN_k

    # Tensorial maximum shear strain rate: gamma_dot_max / 2.
    shear_tensorial = 0.5 * np.sqrt(diff**2 + 4.0 * eps_EN_k**2)

    # Report either engineering or tensorial maximum shear strain rate.
    if engineering_shear:
        shear_out = 2.0 * shear_tensorial
    else:
        shear_out = shear_tensorial

    return {
        "eps_1_hat": tr / 2.0 + shear_tensorial,
        "eps_2_hat": tr / 2.0 - shear_tensorial,
        "theta_hat": 0.5 * np.arctan2(2.0 * eps_EN_k, diff),
        "I1_hat": tr,
        "gamma_max_hat": shear_out,
    }


# ============================================================
# --- Formal delta-method propagation ---
# ============================================================

def compute_derived_quantities_formal(
    eps_dot_hat_elem,
    Q_eps_hat_elem,
    engineering_shear=False,
    eps=1e-20,
):
    n = eps_dot_hat_elem.shape[0]

    out = {
        "eps_1_hat": np.full(n, np.nan),
        "eps_2_hat": np.full(n, np.nan),
        "theta_hat": np.full(n, np.nan),
        "I1_hat": np.full(n, np.nan),
        "gamma_max_hat": np.full(n, np.nan),

        "sigma_eps_1": np.full(n, np.nan),
        "sigma_eps_2": np.full(n, np.nan),
        "sigma_theta": np.full(n, np.nan),
        "sigma_I1": np.full(n, np.nan),
        "sigma_gamma_max": np.full(n, np.nan),

        # One complete local covariance block per element for
        # [epsilon_1, epsilon_2, I1, reported maximum gamma_max_hat].
        "q_hat_order": ("eps_1_hat", "eps_2_hat", "I1_hat", "gamma_max_hat"),
        "Q_q_hat_elem": np.full((n, 4, 4), np.nan),

        "uncertainty_mode": "formal",
        "engineering_shear": engineering_shear,
        "gamma_max_convention": _gamma_max_convention(engineering_shear),
        "label_gamma_max": _gamma_max_label(engineering_shear),
    }

    def prop_sigma(J, C):
        if np.any(~np.isfinite(J)):
            return np.nan
        var = J @ C @ J
        return np.sqrt(max(var, 0.0))

    for k in range(n):
        eps_EE_k = eps_dot_hat_elem[k, 0, 0]
        eps_NN_k = eps_dot_hat_elem[k, 1, 1]
        eps_EN_k = eps_dot_hat_elem[k, 0, 1]

        C = np.asarray(Q_eps_hat_elem[k], dtype=float)
        C = 0.5 * (C + C.T)

        vals = _compute_derived_values(
            eps_EE_k,
            eps_NN_k,
            eps_EN_k,
            engineering_shear=engineering_shear,
        )

        for key in ["eps_1_hat", "eps_2_hat", "theta_hat", "I1_hat", "gamma_max_hat"]:
            out[key][k] = vals[key]

        diff = eps_EE_k - eps_NN_k
        denom = np.sqrt(diff**2 + 4.0 * eps_EN_k**2)

        J_I1 = np.array([1.0, 1.0, 0.0])

        if denom > eps:
            # Derivatives of tensorial maximum shear strain rate.
            dR_dexx = 0.5 * diff / denom
            dR_deyy = -0.5 * diff / denom
            dR_dexy = 2.0 * eps_EN_k / denom

            J_shear_tensorial = np.array([dR_dexx, dR_deyy, dR_dexy])

            if engineering_shear:
                J_gamma_max = 2.0 * J_shear_tensorial
            else:
                J_gamma_max = J_shear_tensorial

            # Principal strain rates always use tensorial maximum shear.
            J_eps_1 = np.array([
                0.5 + dR_dexx,
                0.5 + dR_deyy,
                dR_dexy,
            ])
            J_eps_2 = np.array([
                0.5 - dR_dexx,
                0.5 - dR_deyy,
                -dR_dexy,
            ])

            denom_theta = diff**2 + 4.0 * eps_EN_k**2
            J_theta = np.array([
                -eps_EN_k / denom_theta,
                 eps_EN_k / denom_theta,
                 diff / denom_theta,
            ])
        else:
            # Principal direction and maximum-shear derivatives are undefined
            # at isotropic or near-null strain states.
            J_gamma_max = np.array([np.nan, np.nan, np.nan])
            J_eps_1 = np.array([np.nan, np.nan, np.nan])
            J_eps_2 = np.array([np.nan, np.nan, np.nan])
            J_theta = np.array([np.nan, np.nan, np.nan])

        # Propagate the complete covariance block of
        # [epsilon_1, epsilon_2, I1, gamma_max].
        J_q = np.vstack([J_eps_1, J_eps_2, J_I1, J_gamma_max])
        valid_rows = np.all(np.isfinite(J_q), axis=1)
        Q_quantities = np.full((4, 4), np.nan)

        if np.any(valid_rows):
            B_valid = J_q[valid_rows]
            Q_valid = B_valid @ C @ B_valid.T
            Q_valid = 0.5 * (Q_valid + Q_valid.T)
            Q_quantities[np.ix_(valid_rows, valid_rows)] = Q_valid

        out["Q_q_hat_elem"][k] = Q_quantities

        out["sigma_I1"][k] = prop_sigma(J_I1, C)
        out["sigma_gamma_max"][k] = prop_sigma(J_gamma_max, C)
        out["sigma_eps_1"][k] = prop_sigma(J_eps_1, C)
        out["sigma_eps_2"][k] = prop_sigma(J_eps_2, C)
        out["sigma_theta"][k] = prop_sigma(J_theta, C)

    return out


# ============================================================
# --- Build strain-tensor input from the inversion output ---
# ============================================================

n_elem = len(elements)

eps_dot_hat_elem = np.zeros((n_elem, 2, 2))
eps_dot_hat_elem[:, 0, 0] = eps_EE_hat
eps_dot_hat_elem[:, 1, 1] = eps_NN_hat
eps_dot_hat_elem[:, 0, 1] = eps_EN_hat
eps_dot_hat_elem[:, 1, 0] = eps_EN_hat


# ============================================================
# --- Compute the derived strain-rate quantities ---
# ============================================================

derived_quantities = compute_derived_quantities_formal(
    eps_dot_hat_elem,
    Q_eps_hat_elem,
    engineering_shear=ENGINEERING_SHEAR,
)


# ============================================================
# --- Restore convenient full-field variable names ---
# ============================================================

eps_1_hat = derived_quantities["eps_1_hat"]
eps_2_hat = derived_quantities["eps_2_hat"]
theta_hat = derived_quantities["theta_hat"]

sigma_eps_1 = derived_quantities["sigma_eps_1"]
sigma_eps_2 = derived_quantities["sigma_eps_2"]
sigma_theta = derived_quantities["sigma_theta"]

I1_hat = derived_quantities["I1_hat"]
gamma_max_hat = derived_quantities["gamma_max_hat"]

sigma_I1 = derived_quantities["sigma_I1"]
sigma_gamma_max = derived_quantities["sigma_gamma_max"]

# Complete local 4 x 4 covariance blocks in the documented quantity order.
q_hat_order = derived_quantities["q_hat_order"]
Q_q_hat_elem = derived_quantities["Q_q_hat_elem"]


# ============================================================
# --- Verify the covariance-block diagonals ---
# ============================================================
# Stacking the Jacobians adds cross-covariances but must not change any
# marginal standard deviation.
previous_stds = np.column_stack([
    sigma_eps_1,
    sigma_eps_2,
    sigma_I1,
    sigma_gamma_max,
])
block_stds = np.sqrt(
    np.maximum(np.diagonal(Q_q_hat_elem, axis1=1, axis2=2), 0.0)
)

std_consistency = {}
for j, name in enumerate(q_hat_order):
    matches = np.allclose(
        previous_stds[:, j],
        block_stds[:, j],
        rtol=1e-10,
        atol=1e-14,
        equal_nan=True,
    )
    std_consistency[name] = matches
    print(f"STD consistency [{name:>9s}] : {matches}")

if not all(std_consistency.values()):
    raise AssertionError(
        "At least one marginal standard deviation differs from the "
        "diagonal of Q_q_hat_elem."
    )

label_gamma_max = derived_quantities["label_gamma_max"]
gamma_max_convention = derived_quantities["gamma_max_convention"]

print("Computed full-field strain-rate quantities and propagated uncertainties.")
print(f"Uncertainty propagation mode : {derived_quantities['uncertainty_mode']}")
print(f"Shear convention used        : {derived_quantities['gamma_max_convention']}")
print(f"Shear label                  : {derived_quantities['label_gamma_max']}")
print(f"Number of FEM elements       : {len(I1_hat)}")


STD consistency [eps_1_hat] : True
STD consistency [eps_2_hat] : True
STD consistency [   I1_hat] : True
STD consistency [gamma_max_hat] : True
Computed full-field strain-rate quantities and propagated uncertainties.
Uncertainty propagation mode : formal
Shear convention used        : engineering
Shear label                  : $\hat{\dot{\gamma}}_{\max}$
Number of FEM elements       : 6650


## Export results csv tables


In [6]:
# ============================================================
# --- Build and export the two result tables ---
# ============================================================

# Assign one finite-element identifier to every model element. Elements
# without an observed RUM retain a missing RUM identifier.
n_elem = len(elements)
rum_id = pd.array([pd.NA] * n_elem, dtype="Int64")
rum_id[rum_to_fe] = np.arange(1, len(rum_to_fe) + 1)
fe_id = np.arange(1, n_elem + 1)
xy = np.asarray(mesh["elem_centers"], dtype=float)
Q_q_hat_elem = np.asarray(Q_q_hat_elem, dtype=float)
# Convert velocity gradients from mm/m/yr to microstrain/yr. Variances
# and covariances therefore require the squared scale factor.
scale = 1000.0
scale2 = scale**2

# Element-average strain-rate tensor components and local covariance blocks.
tensor_df = pd.DataFrame({
    "rum_id": rum_id, "fe_id": fe_id,
    "x_fe": xy[:, 0], "y_fe": xy[:, 1],
    "eps_ee": eps_EE_hat * scale, "eps_nn": eps_NN_hat * scale, "eps_en": eps_EN_hat * scale,
    "var_eps_ee": Q_eps_hat_elem[:, 0, 0] * scale2,
    "var_eps_nn": Q_eps_hat_elem[:, 1, 1] * scale2,
    "var_eps_en": Q_eps_hat_elem[:, 2, 2] * scale2,
    "cov_eps_ee_nn": Q_eps_hat_elem[:, 0, 1] * scale2,
    "cov_eps_ee_en": Q_eps_hat_elem[:, 0, 2] * scale2,
    "cov_eps_nn_en": Q_eps_hat_elem[:, 1, 2] * scale2,
})
# Derived quantities and their complete local 4 x 4 covariance blocks.
quantity_df = pd.DataFrame({
    "rum_id": rum_id, "fe_id": fe_id,
    "x_fe": xy[:, 0], "y_fe": xy[:, 1],
    "eps1": eps_1_hat * scale, "eps2": eps_2_hat * scale,
    "i1": I1_hat * scale, "gamma_max": gamma_max_hat * scale,
    "var_eps1": Q_q_hat_elem[:, 0, 0] * scale2,
    "var_eps2": Q_q_hat_elem[:, 1, 1] * scale2,
    "var_i1": Q_q_hat_elem[:, 2, 2] * scale2,
    "var_gamma_max": Q_q_hat_elem[:, 3, 3] * scale2,
    "cov_eps1_eps2": Q_q_hat_elem[:, 0, 1] * scale2,
    "cov_eps1_i1": Q_q_hat_elem[:, 0, 2] * scale2,
    "cov_eps1_gamma_max": Q_q_hat_elem[:, 0, 3] * scale2,
    "cov_eps2_i1": Q_q_hat_elem[:, 1, 2] * scale2,
    "cov_eps2_gamma_max": Q_q_hat_elem[:, 1, 3] * scale2,
    "cov_i1_gamma_max": Q_q_hat_elem[:, 2, 3] * scale2,
})

# Write the result tables. These files are refreshed whenever this cell runs.
tensor_df.to_csv(TENSOR_CSV_PATH, index=False)
quantity_df.to_csv(QUANTITY_CSV_PATH, index=False)

# Add the derived quantities to the in-memory archive. Save the archive only
# when it was produced by a new full inversion; the distributed precomputed
# archive remains unchanged when RUN_FULL_INVERSION is False.
archive["strain_quantities"] = {
    "eps_1_hat": eps_1_hat,
    "eps_2_hat": eps_2_hat,
    "theta_hat": theta_hat,
    "I1_hat": I1_hat,
    "gamma_max_hat": gamma_max_hat,
    "Q_q_hat_elem": Q_q_hat_elem,
    "var_theta_hat": sigma_theta**2,
}
if RUN_FULL_INVERSION:
    with ARCHIVE_PATH.open("wb") as stream:
        pickle.dump(archive, stream, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Updated inversion archive: {ARCHIVE_PATH.resolve()}")
else:
    print(f"Kept precomputed inversion archive unchanged: {ARCHIVE_PATH.resolve()}")

# Record the exported row counts in the accompanying metadata file.
metadata = json.loads(METADATA_JSON_PATH.read_text()) if METADATA_JSON_PATH.exists() else {}
output_csv_tables = metadata.setdefault("output_csv_tables", {})
output_csv_tables.setdefault("strain_tensor", {})["rows"] = len(tensor_df)
output_csv_tables.setdefault("strain_quantities", {})["rows"] = len(quantity_df)
METADATA_JSON_PATH.write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Wrote {len(tensor_df)} tensor rows and {len(quantity_df)} quantity rows.")
print(f"  Strain-tensor table: {TENSOR_CSV_PATH.resolve()}")
print(f"  Strain-quantity table: {QUANTITY_CSV_PATH.resolve()}")
print(f"  Metadata: {METADATA_JSON_PATH.resolve()}")


Kept precomputed inversion archive unchanged: /home/parallels/jakarta_strain_public_workflow/results/jakarta_strain_inversion_archive.pkl
Wrote 6650 tensor rows and 6650 quantity rows.
  Strain-tensor table: /home/parallels/jakarta_strain_public_workflow/results/jakarta_strain_tensor.csv
  Strain-quantity table: /home/parallels/jakarta_strain_public_workflow/results/jakarta_strain_quantities.csv
  Metadata: /home/parallels/jakarta_strain_public_workflow/results/jakarta_strain_metadata.json
